# 11 Topic Word Cloud Analysis

This notebook handles the added Fig11: cluster-specific thematic word clouds for six topic clusters. Inputs are fixed to the new master dataset `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` and the new keyword table `data/NSFC正式增量采集_37个关键词.csv`, reusing the six-topic specification from Fig10/Table1.

Outputs are restricted to:

- `output/figures/Fig11_thematic_wordclouds.svg/pdf/tiff/png`
- `output/tables/11_topic_wordcloud_terms.csv`
- `output/logs/11_wordcloud_text.md`

Fig11 contract:


In [ ]:
# ===== 1. Environment Setup, Paths, and Style =====

# This notebook only reads the new master dataset and the new 37-term keyword table; it does not write back to the data directory.
# Cells 4 and 5 retain Fig11 table and figure export logic; Cell 6 only checks outputs by default, and the log is overwritten only when EXPORT_LOG=True.

from pathlib import Path
from collections import Counter
from datetime import datetime
import math
import os
import re

# In headless environments, give matplotlib a writable cache directory to avoid font-cache errors.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a substitute.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

FIG_FONT_FAMILY = "Times New Roman"


MAIN_DATA_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
KEYWORD_FILENAME = "NSFC正式增量采集_37个关键词.csv"
EXPECTED_MASTER_N = 9222
EXPECTED_KEYWORD_N = 37


def find_project_root(start: Path) -> Path:
    # Find the project root upward from the current directory so the notebook can run from code/ or the project root.
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / MAIN_DATA_FILENAME).exists() and (candidate / "data" / KEYWORD_FILENAME).exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find data/{MAIN_DATA_FILENAME} and data/{KEYWORD_FILENAME} from the current working directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / MAIN_DATA_FILENAME
KEYWORD_PATH = PROJECT_ROOT / "data" / KEYWORD_FILENAME
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
TABLE_DIR = PROJECT_ROOT / "output" / "tables"
LOG_DIR = PROJECT_ROOT / "output" / "logs"
CODE_PATH = PROJECT_ROOT / "code" / "11_topic_word_cloud_analysis.ipynb"

FIG_BASE = FIGURE_DIR / "Fig11_thematic_wordclouds"
TABLE_PATH = TABLE_DIR / "11_topic_wordcloud_terms.csv"
LOG_PATH = LOG_DIR / "11_wordcloud_text.md"

# The log is not written by default to avoid unintentionally overwriting manually reviewed notes when rerunning the notebook.
EXPORT_LOG = False

for folder in [FIGURE_DIR, TABLE_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "savefig.facecolor": "#FFFFFF",
    "savefig.edgecolor": "none",
    "font.family": FIG_FONT_FAMILY,
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 8,
})

TOKENS = {
    "ink": "#1F2430",
    "muted": "#687083",
    "panel_edge": "#D7DBE7",
    "grid": "#EEF0F5",
}

THEME_COLORS = {
    "Remote sensing and urban thermal environment": {"fill": "#AFC7E8", "edge": "#3D5F93", "label": "Remote sensing & thermal"},
    "Street-view and human-scale built environment": {"fill": "#DDA6BF", "edge": "#8A4E70", "label": "Street-view & human scale"},
    "Nighttime light and carbon-energy performance": {"fill": "#E7D37E", "edge": "#8C7A2A", "label": "Nighttime light & carbon-energy"},
    "Multisource data and urban resilience": {"fill": "#B9D7A1", "edge": "#4E7A35", "label": "Multisource & resilience"},
    "AI-enabled urban form and land-use measurement": {"fill": "#E7A07D", "edge": "#93513A", "label": "AI urban form & land use"},
    "Exposure, health and environmental risk": {"fill": "#C7D7F2", "edge": "#5E75A8", "label": "Exposure & risk"},
}
THEME_ORDER = list(THEME_COLORS.keys())

print(f"Project root: {PROJECT_ROOT}")
print(f"Input main data: {DATA_PATH}")
print(f"Input keyword table: {KEYWORD_PATH}")


In [ ]:
# ===== 2. Load the New Master Dataset and Keyword Table, Then Reuse the Fig10 Topic Specification for Record Assignment =====

# matched_* fields are used for Fig10-compatible topic assignment; original Chinese text fields are used for word-cloud term detection.

df = pd.read_csv(DATA_PATH)
keyword_df = pd.read_csv(KEYWORD_PATH)

if len(df) != EXPECTED_MASTER_N:
    raise ValueError(f"Expected {EXPECTED_MASTER_N} records in the new master dataset, got {len(df)}.")

KEYWORD_REQUIRED_COLUMNS = ["term", "term_group"]
missing_keyword_columns = [col for col in KEYWORD_REQUIRED_COLUMNS if col not in keyword_df.columns]
if missing_keyword_columns:
    raise ValueError(f"Missing required keyword-table columns: {missing_keyword_columns}")

keyword_df = keyword_df.copy()
keyword_df["term"] = keyword_df["term"].astype(str).str.strip()
keyword_df["term_group"] = keyword_df["term_group"].astype(str).str.strip()
KEYWORD_TERMS = keyword_df["term"].tolist()
KEYWORD_SET = set(KEYWORD_TERMS)
KEYWORD_ROLES = dict(zip(keyword_df["term"], keyword_df["term_group"]))

if len(KEYWORD_TERMS) != EXPECTED_KEYWORD_N or len(KEYWORD_SET) != EXPECTED_KEYWORD_N:
    raise ValueError(
        f"Expected {EXPECTED_KEYWORD_N} unique controlled keywords, "
        f"got {len(KEYWORD_TERMS)} rows and {len(KEYWORD_SET)} unique terms."
    )

TERM_COLUMNS = [
    "matched_keywords",
    "matched_method_terms",
    "matched_object_terms",
    "matched_performance_terms",
]
RAW_TEXT_COLUMNS = [
    "project_title",
    "abstract_text",
    "keywords_raw",
    "outcomes_text",
    "project_name",
    "project_keywords",
    "project_abstract_cn",
    "project_abstract_en",
    "conclusion_abstract",
]
TEXT_COLUMNS = [col for col in [*RAW_TEXT_COLUMNS, *TERM_COLUMNS] if col in df.columns]
REQUIRED_COLUMNS = [*TERM_COLUMNS, "project_title", "abstract_text", "keywords_raw", "outcomes_text"]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


def split_terms(value):
    # Split matched terms delimited by semicolons, Chinese semicolons, enumeration commas, or pipes.
    if pd.isna(value):
        return []
    text = str(value).replace("；", ";").replace("、", ";").replace("|", ";")
    terms = []
    for term in text.split(";"):
        term = term.strip()
        if term and term.lower() != "nan":
            terms.append(term)
    return terms


observed_matched_terms = set()
for col in TERM_COLUMNS:
    for value in df[col]:
        observed_matched_terms.update(split_terms(value))
unexpected_terms = sorted(observed_matched_terms - KEYWORD_SET)
if unexpected_terms:
    raise ValueError(f"Matched terms outside the 37-keyword table: {unexpected_terms}")


def record_terms(row):
    terms = set()
    for col in TERM_COLUMNS:
        terms.update(split_terms(row[col]))
    return sorted(term for term in terms if term in KEYWORD_SET)


def combine_text(row):
    # Combine Chinese titles, abstracts, keywords, output text, NSFC detail text, and matched fields only for term detection; do not write back to source data.
    parts = []
    for col in TEXT_COLUMNS:
        value = row.get(col, "")
        if pd.notna(value):
            parts.append(str(value))
    return " ".join(parts)


df = df.copy()
df["_terms"] = df.apply(record_terms, axis=1)
df["_combined_text"] = df.apply(combine_text, axis=1)

# Six-topic rule weights compatible with Fig10/Table1. High weights indicate strong topical direction; broad terms receive lower weights.
THEME_WEIGHTS = {
    "Remote sensing and urban thermal environment": {
        "遥感": 3.0, "高分": 3.0, "热岛": 3.0, "热环境": 2.5,
        "气候": 1.4, "GIS": 1.0, "地理信息": 1.0, "绿地": 1.5,
        "生态环境": 1.0, "城市群": 0.6,
    },
    "Street-view and human-scale built environment": {
        "街景": 4.0, "街道": 3.0, "街区": 3.0, "POI": 3.0,
        "LBS": 3.0, "手机信令": 3.0, "轨迹": 3.0, "建成环境": 2.2,
        "大数据": 0.9, "规划": 0.5,
    },
    "Nighttime light and carbon-energy performance": {
        "夜间灯光": 4.0, "碳": 3.2, "能源": 3.2, "建筑": 1.5,
        "规划": 0.7, "大数据": 0.8, "遥感": 0.8, "高分": 0.8,
    },
    "Multisource data and urban resilience": {
        "多源数据": 3.2, "韧性": 3.0, "洪涝": 3.0, "海绵": 3.0,
        "生态环境": 2.0, "气候": 1.2, "GIS": 1.0, "遥感": 1.0,
        "城市群": 0.8,
    },
    "AI-enabled urban form and land-use measurement": {
        "人工智能": 4.0, "机器学习": 3.5, "深度学习": 3.5, "三维": 3.0,
        "LiDAR": 3.0, "城市形态": 3.0, "土地利用": 2.4, "规划": 1.7,
        "城市群": 1.3, "大数据": 1.6, "GIS": 1.2, "建筑": 1.0,
    },
    "Exposure, health and environmental risk": {
        "暴露": 4.0, "空气污染": 4.0, "生态环境": 1.6, "热环境": 1.8,
        "绿地": 1.5, "建筑": 1.0, "气候": 1.0, "洪涝": 1.0,
        "热岛": 0.8,
    },
}

weight_terms = set().union(*(weights.keys() for weights in THEME_WEIGHTS.values()))
missing_weight_terms = sorted(weight_terms - KEYWORD_SET)
if missing_weight_terms:
    raise ValueError(f"Fig10 topic weights reference terms outside the 37-keyword table: {missing_weight_terms}")


def topic_scores(terms):
    terms = set(terms)
    scores = {}
    specific_hits = {}
    for theme in THEME_ORDER:
        weights = THEME_WEIGHTS[theme]
        scores[theme] = sum(weights.get(term, 0.0) for term in terms)
        specific_hits[theme] = sum(1 for term in terms if weights.get(term, 0.0) >= 2.5)
    return scores, specific_hits


def assign_topic(terms):
    scores, specific_hits = topic_scores(terms)
    best_theme = max(
        THEME_ORDER,
        key=lambda theme: (scores[theme], specific_hits[theme], -THEME_ORDER.index(theme)),
    )
    if scores[best_theme] <= 0:
        best_theme = "AI-enabled urban form and land-use measurement"
    return best_theme


df["topic_cluster"] = df["_terms"].map(assign_topic)
cluster_counts = df["topic_cluster"].value_counts().reindex(THEME_ORDER, fill_value=0)
cluster_share = (cluster_counts / len(df) * 100).round(1)
topic_summary_df = pd.DataFrame({
    "topic_id": [f"T{i}" for i in range(1, len(THEME_ORDER) + 1)],
    "topic_cluster": THEME_ORDER,
    "topic_label": [THEME_COLORS[theme]["label"] for theme in THEME_ORDER],
    "topic_N": [int(cluster_counts.loc[theme]) for theme in THEME_ORDER],
    "topic_share_percent": [float(cluster_share.loc[theme]) for theme in THEME_ORDER],
})

print(f"Records analyzed: {len(df):,}")
print(f"Controlled keywords: {len(KEYWORD_TERMS)}")
print(topic_summary_df.to_string(index=False))
assert int(cluster_counts.sum()) == len(df)
assert (cluster_counts > 0).all(), "All six topic clusters should contain records."


In [ ]:
# ===== 3. Chinese Term Detection Constrained by the 37 Keywords and Standard English Vocabulary =====

# Word clouds use the 37 terms in the new keyword table as controlled anchors; primary English labels follow the Fig10 keyword labels.
# variants capture synonymous forms in titles, abstracts, keywords, and output text; high-frequency variants can appear as visible terms but must trace back to one of the 37 controlled keywords.

TERM_LABELS = {
    "遥感": "Remote sensing",
    "大数据": "Big data",
    "机器学习": "Machine learning",
    "手机信令": "Mobile signaling",
    "高分": "High-resolution imagery",
    "建成环境": "Built environment",
    "城市形态": "Urban morphology",
    "规划": "Planning",
    "碳": "Carbon",
    "能源": "Energy",
    "GIS": "GIS",
    "土地利用": "Land use",
    "气候": "Climate",
    "城市群": "Urban agglomeration",
    "生态环境": "Ecological environment",
    "夜间灯光": "Nighttime light",
    "建筑": "Buildings",
    "绿地": "Green space",
    "街景": "Street view",
    "POI": "POI",
    "轨迹": "Mobility traces",
    "街区": "Neighborhood/block",
    "多源数据": "Multisource data",
    "三维": "3D modeling",
    "深度学习": "Deep learning",
    "热岛": "Urban heat island",
    "韧性": "Resilience",
    "热环境": "Thermal environment",
    "暴露": "Exposure",
    "地理信息": "Geoinformation",
    "洪涝": "Flooding",
    "街道": "Street network",
    "海绵": "Sponge city",
    "人工智能": "Artificial intelligence",
    "LiDAR": "LiDAR",
    "LBS": "LBS",
    "空气污染": "Air pollution",
}

PLOT_LABELS = {
    "高分": "High-res imagery",
    "手机信令": "Mobile signaling",
    "城市群": "Urban agglomeration",
    "生态环境": "Ecological environment",
    "夜间灯光": "Nighttime light",
    "轨迹": "Mobility traces",
    "街区": "Neighborhood",
    "多源数据": "Multisource data",
    "人工智能": "AI",
}

TERM_VARIANTS = {
    "遥感": ["遥感", "遥感影像", "遥感数据"],
    "大数据": ["大数据", "城市大数据", "时空大数据"],
    "机器学习": ["机器学习", "随机森林", "支持向量机", "梯度提升", "XGBoost"],
    "手机信令": ["手机信令", "移动信令"],
    "高分": ["高分", "高分辨率", "高分影像", "高分辨率遥感"],
    "建成环境": ["建成环境"],
    "城市形态": ["城市形态", "空间形态", "城市空间形态"],
    "规划": ["规划", "城市规划", "空间规划"],
    "碳": ["碳", "碳排放", "低碳", "碳减排", "碳达峰", "碳中和", "双碳"],
    "能源": ["能源", "能耗", "能源消耗", "能源效率"],
    "GIS": ["GIS", "地理信息系统"],
    "土地利用": ["土地利用", "用地", "土地覆被"],
    "气候": ["气候", "气候变化", "气候适应"],
    "城市群": ["城市群", "都市圈", "城市群地区"],
    "生态环境": ["生态环境", "生态绩效", "生态效应"],
    "夜间灯光": ["夜间灯光", "夜光遥感", "夜光数据", "夜间灯光数据"],
    "建筑": ["建筑", "建筑物", "建筑环境"],
    "绿地": ["绿地", "公园绿地", "绿色空间"],
    "街景": ["街景", "街景图像", "街景影像"],
    "POI": ["POI", "兴趣点"],
    "轨迹": ["轨迹", "出行轨迹", "移动轨迹", "时空轨迹"],
    "街区": ["街区", "社区", "邻里", "居住区"],
    "多源数据": ["多源数据", "多源大数据", "多源时空", "多源遥感"],
    "三维": ["三维", "3D", "三维建模", "三维模型"],
    "深度学习": ["深度学习", "神经网络", "卷积神经网络", "CNN"],
    "热岛": ["热岛", "城市热岛"],
    "韧性": ["韧性", "城市韧性"],
    "热环境": ["热环境", "热舒适", "热暴露"],
    "暴露": ["暴露", "环境暴露", "风险暴露"],
    "地理信息": ["地理信息", "空间信息"],
    "洪涝": ["洪涝", "内涝", "暴雨", "雨洪"],
    "街道": ["街道", "路网", "道路网络", "街道网络"],
    "海绵": ["海绵", "海绵城市"],
    "人工智能": ["人工智能", "AI"],
    "LiDAR": ["LiDAR", "激光雷达", "点云"],
    "LBS": ["LBS", "位置服务", "位置数据"],
    "空气污染": ["空气污染", "大气污染", "PM2.5", "颗粒物"],
}

VARIANT_PLOT_LABELS = {
    "遥感影像": "Remote-sensing imagery",
    "遥感数据": "Remote-sensing data",
    "城市大数据": "Urban big data",
    "时空大数据": "Spatiotemporal big data",
    "随机森林": "Random forest",
    "支持向量机": "SVM",
    "梯度提升": "Gradient boosting",
    "XGBoost": "XGBoost",
    "移动信令": "Mobile signaling data",
    "高分辨率": "High-resolution mapping",
    "高分影像": "High-resolution imagery",
    "高分辨率遥感": "High-resolution remote sensing",
    "空间形态": "Spatial morphology",
    "城市空间形态": "Urban spatial form",
    "城市规划": "Urban planning",
    "空间规划": "Spatial planning",
    "碳排放": "Carbon emissions",
    "低碳": "Low-carbon",
    "碳减排": "Carbon reduction",
    "碳达峰": "Carbon peaking",
    "碳中和": "Carbon neutrality",
    "双碳": "Dual-carbon goals",
    "能耗": "Energy use",
    "能源消耗": "Energy consumption",
    "能源效率": "Energy efficiency",
    "地理信息系统": "GIS systems",
    "用地": "Urban land use",
    "土地覆被": "Land cover",
    "气候变化": "Climate change",
    "气候适应": "Climate adaptation",
    "都市圈": "Metropolitan area",
    "城市群地区": "Urban agglomeration region",
    "生态绩效": "Ecological performance",
    "生态效应": "Ecological effects",
    "夜光遥感": "Nighttime-light remote sensing",
    "夜光数据": "Night-light data",
    "夜间灯光数据": "Nighttime-light data",
    "建筑物": "Building footprints",
    "建筑环境": "Building environment",
    "公园绿地": "Park green space",
    "绿色空间": "Green spaces",
    "街景图像": "Street-view images",
    "街景影像": "Street-view imagery",
    "兴趣点": "Points of interest",
    "出行轨迹": "Travel trajectories",
    "移动轨迹": "Mobility trajectories",
    "时空轨迹": "Spatiotemporal trajectories",
    "街区": "Urban blocks",
    "社区": "Community",
    "邻里": "Neighborhood scale",
    "居住区": "Residential areas",
    "多源大数据": "Multisource big data",
    "多源时空": "Multisource spatiotemporal data",
    "多源遥感": "Multisource remote sensing",
    "3D": "3D data",
    "三维建模": "3D modeling methods",
    "三维模型": "3D models",
    "神经网络": "Neural networks",
    "卷积神经网络": "CNN models",
    "CNN": "CNN",
    "城市热岛": "Urban heat island effect",
    "城市韧性": "Urban resilience",
    "热舒适": "Thermal comfort",
    "热暴露": "Heat exposure",
    "环境暴露": "Environmental exposure",
    "风险暴露": "Risk exposure",
    "空间信息": "Spatial information",
    "内涝": "Urban waterlogging",
    "暴雨": "Rainstorm",
    "雨洪": "Stormwater",
    "路网": "Road network",
    "道路网络": "Road networks",
    "街道网络": "Street networks",
    "海绵城市": "Sponge-city planning",
    "AI": "AI methods",
    "激光雷达": "Laser scanning",
    "点云": "Point clouds",
    "位置服务": "Location-based services",
    "位置数据": "Location data",
    "大气污染": "Atmospheric pollution",
    "PM2.5": "PM2.5",
    "颗粒物": "Particulate matter",
}

missing_keyword_labels = sorted(KEYWORD_SET - set(TERM_LABELS))
missing_keyword_variants = sorted(KEYWORD_SET - set(TERM_VARIANTS))
if missing_keyword_labels or missing_keyword_variants:
    raise ValueError(
        f"Missing keyword metadata: labels={missing_keyword_labels}, variants={missing_keyword_variants}"
    )

TERM_LEXICON = {
    term: {
        "label": TERM_LABELS[term],
        "plot": PLOT_LABELS.get(term, TERM_LABELS[term]),
        "term_group": KEYWORD_ROLES[term],
        "variants": TERM_VARIANTS[term],
    }
    for term in KEYWORD_TERMS
}

# Slightly downweight high-frequency broad cross-topic terms from Fig10 without removing them, preserving semantic context.
BROAD_TERM_PENALTY = {
    "规划": 0.70,
    "大数据": 0.78,
    "GIS": 0.82,
    "气候": 0.82,
    "建筑": 0.82,
}

# Visible English labels must be unique; otherwise synonyms will split in the figure.
plot_labels = [item["plot"] for item in TERM_LEXICON.values()]
duplicates = [label for label, count in Counter(plot_labels).items() if count > 1]
if duplicates:
    raise ValueError(f"Duplicated plot labels: {duplicates}")


def has_variant(text, variants):
    # Detect Chinese terms by substring; detect English abbreviations case-insensitively.
    text = str(text)
    text_lower = text.lower()
    for variant in variants:
        v = str(variant)
        if re.fullmatch(r"[A-Za-z0-9.+-]+", v):
            if re.search(rf"(?<![A-Za-z0-9]){re.escape(v.lower())}(?![A-Za-z0-9])", text_lower):
                return True
        elif v in text:
            return True
    return False


def detected_terms(row):
    text = row["_combined_text"]
    terms = set(row["_terms"])
    for term, info in TERM_LEXICON.items():
        if term in terms or has_variant(text, info["variants"]):
            terms.add(term)
    return sorted(term for term in terms if term in TERM_LEXICON)


df["_lexical_terms"] = df.apply(detected_terms, axis=1)
lexical_frequency = Counter(term for terms in df["_lexical_terms"] for term in set(terms))
print(f"Controlled keyword terms in table: {len(TERM_LEXICON)}")
print(f"Controlled keyword terms detected in records: {len(lexical_frequency)}")
print("Top controlled keyword terms:")
for term, count in lexical_frequency.most_common(15):
    print(f"- {TERM_LEXICON[term]['label']}: {count}")


In [ ]:
# ===== 4. Compute Cluster-Specific Word-Cloud Weights and Write the Source Data Table =====

# Running this cell overwrites output/tables/11_topic_wordcloud_terms.csv; it is retained for explicit reruns by later Fig11 table/log workers.
# Weights are not simple total frequencies; they are cluster-specific log-odds times record frequency times cross-cluster specificity.
# Count each term by record-level presence to avoid over-amplifying repeated words in long abstracts.
# Visible terms include primary labels for the 37 controlled keywords plus high-frequency controlled variant labels actually matched in the new master-table text.

N = len(df)
if N != EXPECTED_MASTER_N:
    raise ValueError(f"Expected N={EXPECTED_MASTER_N} for Fig11 scoring, got N={N}.")

K = len(THEME_ORDER)
VARIANT_MIN_CLUSTER_RECORDS = 4
MAX_TERMS_IN_SOURCE_TABLE_PER_TOPIC = 70
VARIANT_SCORE_WEIGHT = 0.74

cluster_term_lists = {
    theme: df.loc[df["topic_cluster"].eq(theme), "_lexical_terms"].tolist()
    for theme in THEME_ORDER
}
term_total_counts = Counter(term for terms in df["_lexical_terms"] for term in set(terms))
term_cluster_presence = Counter()
for term in TERM_LEXICON:
    term_cluster_presence[term] = sum(
        1 for term_lists in cluster_term_lists.values()
        if any(term in terms for terms in term_lists)
    )


def log_odds_cluster(n_tc, n_c, n_t, n_total, alpha=0.5):
    # Monroe-style smoothed log odds for one term in one topic cluster versus the rest of the corpus.
    a = n_tc + alpha
    b = n_c - n_tc + alpha
    c = n_t - n_tc + alpha
    d = (n_total - n_c) - (n_t - n_tc) + alpha
    return math.log(a / b) - math.log(c / d)


def detected_variant_hits(row):
    text = row["_combined_text"]
    hits = []
    for term, info in TERM_LEXICON.items():
        for variant in info["variants"]:
            if variant == term:
                continue
            if variant not in VARIANT_PLOT_LABELS:
                continue
            if has_variant(text, [variant]):
                hits.append((term, variant))
    return hits


df["_variant_hits"] = df.apply(detected_variant_hits, axis=1)
variant_total_counts = Counter(pair for hits in df["_variant_hits"] for pair in set(hits))
variant_cluster_presence = Counter()
for pair in variant_total_counts:
    variant_cluster_presence[pair] = sum(
        1 for theme in THEME_ORDER
        if any(pair in hits for hits in df.loc[df["topic_cluster"].eq(theme), "_variant_hits"])
    )


def make_term_row(theme_index, theme, n_c, topic_share_percent, term, variant, entry_type, label, n_tc, n_t, presence_count, score_weight=1.0):
    cluster_share = n_tc / n_c
    corpus_share = n_t / N
    log_odds = log_odds_cluster(n_tc, n_c, n_t, N)
    idf = math.log((1 + K) / (1 + presence_count)) + 1
    score = max(log_odds, 0.08) * math.log1p(n_tc) * idf
    score *= BROAD_TERM_PENALTY.get(term, 1.0) * score_weight
    info = TERM_LEXICON[term]
    return {
        "topic_id": f"T{theme_index}",
        "topic_cluster": theme,
        "topic_label": THEME_COLORS[theme]["label"],
        "topic_N": n_c,
        "topic_share_percent": topic_share_percent,
        "entry_type": entry_type,
        "term_zh": term,
        "term_variant_zh": variant,
        "term_group": KEYWORD_ROLES[term],
        "term_en": info["label"],
        "plot_label": label,
        "cluster_records": int(n_tc),
        "corpus_records": int(n_t),
        "cluster_share_percent": round(cluster_share * 100, 2),
        "corpus_share_percent": round(corpus_share * 100, 2),
        "cluster_presence_count": int(presence_count),
        "log_odds": round(log_odds, 4),
        "specificity_idf": round(idf, 4),
        "wordcloud_score": round(score, 4),
    }


rows = []
for theme_index, theme in enumerate(THEME_ORDER, start=1):
    n_c = int(cluster_counts.loc[theme])
    min_cluster_count = 1 if n_c < 20 else 2
    topic_share_percent = round(n_c / N * 100, 2)
    term_lists = cluster_term_lists[theme]
    topic_rows = []

    for term, info in TERM_LEXICON.items():
        n_tc = sum(1 for terms in term_lists if term in terms)
        if n_tc < min_cluster_count:
            continue
        n_t = term_total_counts.get(term, 0)
        if n_t == 0:
            continue
        topic_rows.append(make_term_row(
            theme_index, theme, n_c, topic_share_percent,
            term=term,
            variant=term,
            entry_type="controlled_keyword",
            label=info["plot"],
            n_tc=n_tc,
            n_t=n_t,
            presence_count=term_cluster_presence[term],
        ))

    theme_variant_hits = df.loc[df["topic_cluster"].eq(theme), "_variant_hits"].tolist()
    seen_labels = {row["plot_label"] for row in topic_rows}
    for (term, variant), n_t in variant_total_counts.items():
        n_tc = sum(1 for hits in theme_variant_hits if (term, variant) in hits)
        if n_tc < VARIANT_MIN_CLUSTER_RECORDS:
            continue
        label = VARIANT_PLOT_LABELS[variant]
        if label in seen_labels:
            continue
        topic_rows.append(make_term_row(
            theme_index, theme, n_c, topic_share_percent,
            term=term,
            variant=variant,
            entry_type="controlled_variant",
            label=label,
            n_tc=n_tc,
            n_t=n_t,
            presence_count=variant_cluster_presence[(term, variant)],
            score_weight=VARIANT_SCORE_WEIGHT,
        ))
        seen_labels.add(label)

    topic_rows = sorted(
        topic_rows,
        key=lambda row: (row["wordcloud_score"], row["cluster_records"], row["plot_label"]),
        reverse=True,
    )[:MAX_TERMS_IN_SOURCE_TABLE_PER_TOPIC]
    rows.extend(topic_rows)

term_df = pd.DataFrame(rows)
if term_df.empty:
    raise RuntimeError("No word-cloud terms were detected.")
term_df = term_df.sort_values(
    ["topic_id", "wordcloud_score", "cluster_records", "plot_label"],
    ascending=[True, False, False, True],
).reset_index(drop=True)
term_df["rank_in_topic"] = term_df.groupby("topic_id").cumcount() + 1
term_df = term_df[[
    "topic_id", "topic_cluster", "topic_label", "topic_N", "topic_share_percent", "rank_in_topic",
    "entry_type", "term_zh", "term_variant_zh", "term_group", "term_en", "plot_label",
    "cluster_records", "corpus_records", "cluster_share_percent", "corpus_share_percent", "cluster_presence_count",
    "log_odds", "specificity_idf", "wordcloud_score",
]]
term_df.to_csv(TABLE_PATH, index=False, encoding="utf-8-sig")

print(f"Word-cloud source table: {TABLE_PATH.relative_to(PROJECT_ROOT)}")
print(term_df.groupby(["topic_id", "topic_label"]).size().to_string())
print(term_df.groupby("topic_id").head(8).to_string(index=False))


In [ ]:
# ===== 5. Draw Fig11: Dense Open Thematic Word Clouds =====

# Running this cell overwrites output/figures/Fig11_thematic_wordclouds.*; it is retained for explicit reruns by later Fig11 figure workers.
# Follow the visual form of three-panel word-cloud examples: open white background, panel titles, more words, multiple font sizes, multiple colors, and a few vertical terms.
# Estimate text size with PIL, but draw final text with matplotlib text so English labels remain editable in SVG/PDF.

from PIL import ImageFont


def hex_to_rgb(color):
    color = color.lstrip("#")
    return np.array([int(color[i:i+2], 16) for i in (0, 2, 4)]) / 255.0


def rgb_to_hex(rgb):
    rgb = np.clip(np.asarray(rgb), 0, 1)
    return "#" + "".join(f"{int(round(v * 255)):02X}" for v in rgb)


def blend(color, target="#FFFFFF", amount=0.35):
    return rgb_to_hex(hex_to_rgb(color) * (1 - amount) + hex_to_rgb(target) * amount)


PANEL_PALETTES = {
    "Remote sensing and urban thermal environment": ["#2E5078", "#5578A3", "#8A6A3F", "#B75F5B", "#5F6773"],
    "Street-view and human-scale built environment": ["#7B3F68", "#9D5F86", "#5E6C7A", "#B87466", "#8B7A55"],
    "Nighttime light and carbon-energy performance": ["#7B6C1D", "#9B7F2D", "#5F5F5F", "#B56A4E", "#6A7E9A"],
    "Multisource data and urban resilience": ["#3E6D31", "#5A8B4A", "#54707B", "#8A6A3F", "#5F5F5F"],
    "AI-enabled urban form and land-use measurement": ["#80452F", "#A05B40", "#5F5F5F", "#876B45", "#5C6F85"],
    "Exposure, health and environmental risk": ["#516A9D", "#6F7FBA", "#5F5F5F", "#2F8A94", "#8B6B45"],
}

MAX_WORDS_PER_PANEL = 60
MIN_FONT = 3.8
MAX_FONT = 24.5
DPI_FOR_MEASURE = 132
WORD_PADDING = 0.0035
PANEL_LABEL_X = 0.035
PANEL_TITLE_X = 0.070
PANEL_HEADER_Y = 0.965
PANEL_TITLE_FONT_SIZE = 15.8
CLOUD_BOUNDS = (0.020, 0.025, 0.980, 0.855)


def scale_font_sizes(scores, min_size=MIN_FONT, max_size=MAX_FONT):
    values = np.asarray(scores, dtype=float)
    if len(values) == 0:
        return []
    if values.max() == values.min():
        return [0.5 * (min_size + max_size)] * len(values)
    scaled = (np.sqrt(values) - np.sqrt(values.min())) / (np.sqrt(values.max()) - np.sqrt(values.min()))
    sizes = min_size + scaled * (max_size - min_size)
    if len(sizes) > 18:
        sizes[18:] = np.minimum(sizes[18:], 7.0)
    if len(sizes) > 36:
        sizes[36:] = np.minimum(sizes[36:], 5.4)
    return sizes.tolist()


def find_times_font_path():
    for path in TIMES_NEW_ROMAN_PATHS:
        if path.exists() and "Bold" not in path.name and "Italic" not in path.name:
            return path
    return Path(font_manager.findfont("Times New Roman", fallback_to_default=False))


TIMES_FONT_PATH = find_times_font_path()


def text_extent_axes(label, font_size_pt, angle, ax_width_in, ax_height_in):
    # PIL font sizes use pixels; convert points to pixels using the measurement DPI and then to axes fractions.
    font_px = max(5, int(round(font_size_pt * DPI_FOR_MEASURE / 72)))
    font = ImageFont.truetype(str(TIMES_FONT_PATH), font_px)
    bbox = font.getbbox(str(label))
    width_px = max(1, bbox[2] - bbox[0])
    height_px = max(1, bbox[3] - bbox[1])
    width_in = width_px / DPI_FOR_MEASURE
    height_in = height_px / DPI_FOR_MEASURE
    width_axes = width_in / ax_width_in
    height_axes = height_in / ax_height_in
    if angle in {90, -90}:
        width_axes, height_axes = height_axes, width_axes
    return width_axes, height_axes


def rect_from_center(x, y, w, h, pad=WORD_PADDING):
    return (x - w / 2 - pad, y - h / 2 - pad, x + w / 2 + pad, y + h / 2 + pad)


def rect_inside(rect, bounds=CLOUD_BOUNDS):
    x0, y0, x1, y1 = rect
    bx0, by0, bx1, by1 = bounds
    return x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1


def rect_overlap(a, b):
    return not (a[2] <= b[0] or a[0] >= b[2] or a[3] <= b[1] or a[1] >= b[3])


def can_place(rect, placed_rects):
    return rect_inside(rect) and not any(rect_overlap(rect, other) for other in placed_rects)


def candidate_positions(panel_index, rank, n_random=1600):
    # Place large words near the center first, then fill the cloud with spiral and random positions.
    anchors = [
        (0.500, 0.530), (0.355, 0.635), (0.665, 0.630), (0.505, 0.405),
        (0.235, 0.520), (0.775, 0.515), (0.335, 0.755), (0.665, 0.755),
        (0.315, 0.285), (0.685, 0.285), (0.160, 0.665), (0.840, 0.665),
    ]
    if rank <= len(anchors):
        yield anchors[rank - 1]
    rng = np.random.default_rng(202607 + panel_index * 173 + rank * 19)
    cx, cy = 0.50, 0.490
    golden = math.pi * (3 - math.sqrt(5))
    for i in range(1, 520):
        radius = 0.0165 * math.sqrt(i)
        angle = i * golden + 0.37 * panel_index + 0.11 * rank
        x = cx + radius * math.cos(angle)
        y = cy + 0.72 * radius * math.sin(angle)
        if CLOUD_BOUNDS[0] < x < CLOUD_BOUNDS[2] and CLOUD_BOUNDS[1] < y < CLOUD_BOUNDS[3]:
            yield x, y
    for _ in range(n_random):
        yield rng.uniform(CLOUD_BOUNDS[0], CLOUD_BOUNDS[2]), rng.uniform(CLOUD_BOUNDS[1], CLOUD_BOUNDS[3])


def choose_rotation(rank, label):
    # Keep large words horizontal; rotate a small share of small words to mimic vertical edge terms in reference figures.
    if rank <= 12:
        return 0
    if len(label) <= 18 and rank % 8 in {0, 3, 6}:
        return 90
    return 0


def choose_color(theme, rank):
    palette = PANEL_PALETTES[theme]
    if rank <= 4:
        return palette[0]
    return palette[(rank - 1) % len(palette)]


def panel_title(label, n):
    # Prefer single-line titles; allow natural wrapping for longer titles.
    return f"{label} (n={int(n)})"


def draw_dense_wordcloud(ax, fig, panel_terms, theme, panel_index):
    colors = THEME_COLORS[theme]
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_axis_off()

    title = panel_title(colors["label"], cluster_counts.loc[theme])
    ax.text(PANEL_LABEL_X, PANEL_HEADER_Y, chr(96 + panel_index), ha="left", va="top", fontsize=PANEL_TITLE_FONT_SIZE, fontweight="bold", fontfamily=FIG_FONT_FAMILY, color="#111111", transform=ax.transAxes)
    ax.text(PANEL_TITLE_X, PANEL_HEADER_Y, title, ha="left", va="top", fontsize=PANEL_TITLE_FONT_SIZE, fontweight="bold", fontfamily=FIG_FONT_FAMILY, color="#111111", transform=ax.transAxes)

    if panel_terms.empty:
        ax.text(0.5, 0.45, "No detected terms", ha="center", va="center", fontsize=9, fontfamily=FIG_FONT_FAMILY, color=TOKENS["muted"])
        return []

    fig_w, fig_h = fig.get_size_inches()
    ax_pos = ax.get_position()
    ax_width_in = fig_w * ax_pos.width
    ax_height_in = fig_h * ax_pos.height

    top_terms = panel_terms.head(MAX_WORDS_PER_PANEL).copy()
    top_terms["font_size"] = scale_font_sizes(top_terms["wordcloud_score"].tolist())
    placed_rects = []
    placed_terms = []

    for rank, (_, row) in enumerate(top_terms.iterrows(), start=1):
        label = str(row["plot_label"])
        base_size = float(row["font_size"])
        angle = choose_rotation(rank, label)
        color = choose_color(theme, rank)
        font_weight = "bold" if rank <= 8 else "semibold" if rank <= 22 else "normal"
        placed = False
        for shrink in [1.00, 0.93, 0.86, 0.78, 0.70, 0.62, 0.54, 0.48]:
            font_size = max(MIN_FONT, base_size * shrink)
            w, h = text_extent_axes(label, font_size, angle, ax_width_in, ax_height_in)
            for x, y in candidate_positions(panel_index, rank):
                rect = rect_from_center(x, y, w, h)
                if can_place(rect, placed_rects):
                    ax.text(
                        x, y, label,
                        ha="center", va="center",
                        fontsize=font_size,
                        fontweight=font_weight,
                        fontfamily=FIG_FONT_FAMILY,
                        color=color,
                        rotation=angle,
                        rotation_mode="anchor",
                        transform=ax.transAxes,
                        zorder=3,
                    )
                    placed_rects.append(rect)
                    placed_terms.append(label)
                    placed = True
                    break
            if placed:
                break
    return placed_terms


fig, axes = plt.subplots(2, 3, figsize=(15.2, 7.85), facecolor="white")
axes = axes.ravel()
fig.subplots_adjust(left=0.018, right=0.992, top=0.958, bottom=0.030, wspace=0.040, hspace=0.052)

placed_summary = {}
for index, theme in enumerate(THEME_ORDER, start=1):
    panel_terms = term_df[term_df["topic_cluster"].eq(theme)].sort_values("rank_in_topic")
    placed_summary[theme] = draw_dense_wordcloud(axes[index - 1], fig, panel_terms, theme, index)

for ext in ["svg", "pdf", "png", "tiff"]:
    output_path = FIG_BASE.with_suffix(f".{ext}")
    save_kwargs = {"bbox_inches": "tight", "facecolor": "white", "pad_inches": 0.03}
    if ext in {"png", "tiff"}:
        save_kwargs["dpi"] = 600
    if ext == "tiff":
        save_kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(output_path, **save_kwargs)

plt.close(fig)

for suffix in [".svg", ".pdf", ".tiff", ".png"]:
    path = FIG_BASE.with_suffix(suffix)
    print(f"{path.relative_to(PROJECT_ROOT)}: {path.stat().st_size} bytes")

for theme, labels in placed_summary.items():
    print(f"{THEME_COLORS[theme]['label']}: {len(labels)} labels placed")


In [ ]:
# ===== 6. Write the Fig11 Log by Switch and Run Basic Output Checks =====

# EXPORT_LOG defaults to False; set it explicitly to True and rerun this cell when output/logs/11_wordcloud_text.md needs to be refreshed.
# Even when the log is not written, this cell still checks that Fig11 figure files, the CSV, and the existing log are present and non-empty.

topic_ids = [f"T{i}" for i in range(1, len(THEME_ORDER) + 1)]
topic_row_counts = term_df.groupby("topic_id").size()
visible_counts = term_df.groupby("topic_id").head(MAX_WORDS_PER_PANEL).groupby("topic_id").size()
topic_rows_summary = ", ".join(f"{topic_id}={int(topic_row_counts.get(topic_id, 0))}" for topic_id in topic_ids)
visible_summary = ", ".join(f"{topic_id}={int(visible_counts.get(topic_id, 0))}" for topic_id in topic_ids)

topic_lines = []
for theme_index, theme in enumerate(THEME_ORDER, start=1):
    topic_id = f"T{theme_index}"
    topic_lines.append(
        f"| {topic_id} | {theme} | {int(cluster_counts.loc[theme])} | "
        f"{float(cluster_share.loc[theme]):.1f}% | {int(topic_row_counts.get(topic_id, 0))} | "
        f"{int(visible_counts.get(topic_id, 0))} |"
    )

top_lines = []
for theme_index, theme in enumerate(THEME_ORDER, start=1):
    topic_id = f"T{theme_index}"
    visible_sub = term_df[term_df["topic_cluster"].eq(theme)].head(MAX_WORDS_PER_PANEL)
    sub = visible_sub.head(8)
    terms_text = "; ".join(
        f"{row.plot_label} (n={int(row.cluster_records)}, score={float(row.wordcloud_score):.2f})"
        for row in sub.itertuples()
    )
    top_lines.append(
        f"- {topic_id} {theme} (topic_N={int(cluster_counts.loc[theme])}, visible labels={len(visible_sub)}): "
        f"{terms_text}"
    )

log_text = f"""# Fig11 thematic word-cloud analysis

Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} CST

## Input and scope

- Corpus size: N={len(df)} records (expected N={EXPECTED_MASTER_N}).
- Topic structure: {len(THEME_ORDER)} themes, with topic_N and rounded shares aligned to Table 1.
- Vocabulary scope: {len(TERM_LEXICON)} unique controlled English keywords represented across {len(term_df)} topic-term rows in `output/tables/11_topic_wordcloud_terms.csv`.
- Current source rows by topic: {topic_rows_summary}.
- Visible word-count by panel: {visible_summary}; T1-T5 use the 60-word cap and T6 has 59 available terms.
- Text fields used for controlled-term detection: {', '.join(TEXT_COLUMNS)}
- Word-cloud labels are controlled English terms mapped from Chinese source terms; they are not raw machine translations.
- Fig11 typography/export: Times New Roman; SVG, PDF, TIFF, and PNG are the required four output formats.

## Topic denominators

| Topic | Theme | topic_N | Share | Source rows | Visible terms |
|---|---|---:|---:|---:|---:|

{chr(10).join(topic_lines)}

## Weighting and layout

Each controlled keyword family and recorded variant is counted at record level within each topic cluster. Word-cloud size follows the `wordcloud_score` column, combining smoothed cluster-specific log-odds, log record frequency and cross-cluster specificity. Broad cross-topic terms are retained with lower scores rather than removed.

The rendered Fig11 uses six white-background thematic panels with Times New Roman text, Table 1-consistent panel titles, muted theme-specific colors and dense open label placement. The table stores all {len(term_df)} source rows, including topic_N and topic_share_percent for traceability; the figure renders 60 visible labels for T1-T5 and 59 for T6.

## Top lexical signatures by topic

{chr(10).join(top_lines)}

## Files checked

- `output/figures/Fig11_thematic_wordclouds.svg`
- `output/figures/Fig11_thematic_wordclouds.pdf`
- `output/figures/Fig11_thematic_wordclouds.tiff`
- `output/figures/Fig11_thematic_wordclouds.png`
- `output/tables/11_topic_wordcloud_terms.csv`
- `output/logs/11_wordcloud_text.md`

## Notebook write behavior

The notebook writes this log only when `EXPORT_LOG=True`; default runs leave the existing log unchanged and only print output status.
"""
if EXPORT_LOG:
    LOG_PATH.write_text(log_text, encoding="utf-8")
    print(f"Log written: {LOG_PATH.relative_to(PROJECT_ROOT)}")
else:
    print(f"EXPORT_LOG=False; existing log left unchanged: {LOG_PATH.relative_to(PROJECT_ROOT)}")

status_rows = []
for path in [
    FIG_BASE.with_suffix(".svg"),
    FIG_BASE.with_suffix(".pdf"),
    FIG_BASE.with_suffix(".tiff"),
    FIG_BASE.with_suffix(".png"),
    TABLE_PATH,
    LOG_PATH,
]:
    status_rows.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "nonzero": bool(path.exists() and path.stat().st_size > 0),
    })
status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))
if not status_df["nonzero"].all():
    raise RuntimeError("Some Fig11 outputs are missing or zero bytes.")
